# 01 — Diagnóstico de la fuga de información
**Paper Cañete · Royal Society Open Science**

Este notebook demuestra, con los datos originales, las dos fugas que inflaron las métricas de la tesis:
1. **`dist_sitio_mas_cercano` codifica la etiqueta** (por construcción: etiquetado con buffer de 30 m + distancia al mismo archivo de sitios).
2. **SMOTE aplicado antes de la partición** → el conjunto de prueba contiene datos sintéticos.

Los datos viven en `../datos/`. Tiempo total de ejecución: ~2–4 min.

In [ ]:
import pandas as pd, numpy as np, warnings
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

df = pd.read_csv("../datos/grilla_modelo2_620k.csv.gz", low_memory=False)
df = df.dropna(subset=['altitud','pendiente','dist_acequia','dist_rio']).reset_index(drop=True)
y = df['label'].values
print(f"Celdas: {len(df):,} | positivas: {int(y.sum())} | prevalencia: {y.mean():.5%}")

## 1. La variable que es la etiqueta

En `Grilla.ipynb` (código original de la tesis):
```python
# Celda 2 — etiquetado
buffer_sitios["geometry"] = buffer_sitios.geometry.buffer(30)
grilla_gdf["label"] = grilla_gdf.geometry.apply(
    lambda p: buffer_sitios.geometry.contains(p).any()).astype(int)

# Celda 7 — la variable
sitios_tree = cKDTree(sitios[['x','y']].values)
distancias, _ = sitios_tree.query(grilla[['x','y']].values, k=1)
grilla['dist_sitio_mas_cercano'] = distancias
```
Por construcción: `label = 1 ⟺ dist_sitio_mas_cercano ≤ 30`. Lo verificamos:

In [ ]:
print(df.groupby('label')['dist_sitio_mas_cercano'].agg(['min','max']).to_string())

# Un umbral simple en 30 m clasifica perfecto — sin ningún modelo:
pred_umbral = (df['dist_sitio_mas_cercano'] <= 30).astype(int)
print(f"\nAccuracy del 'clasificador' umbral<=30m: {(pred_umbral==y).mean():.6f}")
print(f"Recall: {pred_umbral[y==1].mean():.6f} | Precisión: {y[pred_umbral==1].mean():.6f}")

In [ ]:
# Figura 'anatomía de la fuga': histograma de dist_sitio por clase
fig, ax = plt.subplots(figsize=(8,4.5))
ax.hist(df.loc[y==0,'dist_sitio_mas_cercano'], bins=np.logspace(0,4.5,60),
        alpha=.6, label='label = 0 (sin registro)', color='steelblue')
ax.hist(df.loc[y==1,'dist_sitio_mas_cercano'], bins=np.logspace(0,4.5,60),
        alpha=.85, label='label = 1 (con sitio)', color='firebrick')
ax.axvline(30, color='k', ls='--', lw=1.5, label='30 m (buffer de etiquetado)')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('dist_sitio_mas_cercano (m, escala log)'); ax.set_ylabel('celdas (log)')
ax.legend(); ax.set_title('La variable dist_sitio codifica la etiqueta por construcción')
plt.tight_layout(); plt.savefig('fig_fuga_dist_sitio.png', dpi=200)
plt.show()

## 2. Reproducción del pipeline original (SMOTE → split)

Reproduce la Tabla 1 de la tesis: el test resultante tiene **372,060 celdas con balance ~50/50**
(en un valle con 0.03% de prevalencia) — la evidencia de que se evaluó sobre datos sintéticos.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from imblearn.over_sampling import SMOTE

FEAT = ['altitud','pendiente','dist_acequia','dist_rio','dist_sitio_mas_cercano']
X = df[FEAT].values
Xr, yr = SMOTE(random_state=42).fit_resample(X, y)           # <- SMOTE ANTES del split (error original)
Xtr, Xte, ytr, yte = train_test_split(Xr, yr, test_size=0.3, random_state=42)
m = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1).fit(Xtr, ytr)
pred = m.predict(Xte)
print(f"Test: {len(yte):,} celdas | balance: {np.bincount(yte)}  <- ~50/50 = sintético")
print(f"accuracy={accuracy_score(yte,pred):.5f}  precision={precision_score(yte,pred):.5f}  "
      f"recall={recall_score(yte,pred):.5f}  f1={f1_score(yte,pred):.5f}")

## Conclusión
- El 0.9995 de la tesis se explica por **dos fugas superpuestas**: la variable-etiqueta y el test sintético.
- Toda evaluación honesta debe: (a) excluir `dist_sitio_mas_cercano` y coordenadas, (b) aplicar SMOTE
  **solo dentro del entrenamiento**, (c) usar validación espacial por bloques (→ notebook 02).